# Object-Agnostic Prompt Training (MVTec + VisA)

This notebook runs the complete **per-source-dataset shallow prompt-training pipeline** on Kaggle. It trains one normal/abnormal prompt pair for MVTec and one for VisA, using the AnomalyCLIP image cross-entropy plus mask-based focal/Dice objective. CLIP stays frozen; deep text prompting and DPAM are disabled.

Before running: enable a **GPU**, attach the MVTec AD and VisA datasets, and enable Internet if the two source repositories and CLIP weights are not attached as Kaggle datasets. Edit only the settings cell below.

In [ ]:
# ========================= USER SETTINGS =========================
PROJECT_GIT_URL = "https://github.com/Parsagh05/object-agnostic-prompt-training.git"  # GitHub repository containing this prompt-training pipeline.
PROJECT_BRANCH = "main"  # Repository branch Kaggle should clone.
PROJECT_SOURCE = None  # Leave None to clone GitHub; otherwise provide an attached project folder or ZIP under /kaggle/input.

ANOMALYCLIP_GIT_URL = "https://github.com/zqhang/AnomalyCLIP.git"  # Official AnomalyCLIP source used to load public CLIP.
ANOMALYCLIP_COMMIT = "3911738c0867544f545a076ad78f3f11d9ecbfdf"  # Pinned revision used by this implementation for reproducibility.
ANOMALYCLIP_SOURCE = None  # Leave None to clone GitHub; otherwise provide an attached AnomalyCLIP folder or ZIP.

DATASETS = "all"  # Choose "all", "mvtec", or "visa"; "all" trains two separate checkpoints.
MVTEC_ROOT = "/kaggle/input/mvtec-ad/mvtec_anomaly_detection"  # Root created by the alirezasalehy/mvtec-ad Kaggle dataset.
VISA_ROOT = "/kaggle/input/visa-ad/VisA_20220922"  # Root created by the alirezasalehy/visa-ad Kaggle dataset.
MVTEC_TRAINING_MANIFEST = None  # Recommended: None recreates the prior balanced 50/50 split with seed 111; set a CSV only to reuse saved attack_train IDs.
VISA_TRAINING_MANIFEST = None  # Recommended: None recreates the same VisA split; a shared prior CSV could also be supplied here.

CLIP_WEIGHTS = None  # Leave None to download ViT-L/14@336px; set an attached .pt path when Kaggle Internet is off.
OUTPUT_ROOT = "/kaggle/working/object_agnostic_prompt_training/artifacts/prompts"  # Folder for checkpoints, histories, configs, and manifests.
BATCH_SIZE = 2  # Safer for a 16 GB Kaggle T4; try 8 for more speed if it does not cause CUDA out-of-memory.
SHOW_PROGRESS = True  # Show tqdm progress bars for all epochs and batches.
OVERWRITE = False  # Keep False to protect finished artifacts; set True only for an intentional rerun.
RUN_TRAINING = True  # Set False to run installation and data validation without starting the 15 training epochs.
# ================================================================


In [ ]:
# Prepare repositories and Python dependencies.
from pathlib import Path
import os, shutil, subprocess, sys, zipfile
import torch

assert Path("/kaggle/working").is_dir(), "This notebook is intended for Kaggle."
if not torch.cuda.is_available():
    raise RuntimeError("Enable a Kaggle GPU before training ViT-L/14@336px.")
print("GPU:", torch.cuda.get_device_name(0))

WORK_ROOT = Path("/kaggle/working/object_agnostic_prompt_training")
PROJECT_ROOT = WORK_ROOT / "project"
ANOMALYCLIP_ROOT = WORK_ROOT / "AnomalyCLIP"
WORK_ROOT.mkdir(parents=True, exist_ok=True)

def materialize_source(source, destination, git_url, branch_or_commit, *, is_commit=False):
    destination = Path(destination)
    if destination.exists():
        print("Reusing:", destination)
    elif source:
        source_path = Path(source)
        if source_path.is_dir():
            shutil.copytree(source_path, destination)
        elif source_path.is_file() and source_path.suffix.lower() == ".zip":
            destination.mkdir(parents=True)
            with zipfile.ZipFile(source_path) as archive:
                archive.extractall(destination)
            children = [p for p in destination.iterdir() if p.is_dir()]
            if len(children) == 1 and not (destination / "pyproject.toml").exists() and not (destination / "AnomalyCLIP_lib").exists():
                nested = children[0]
                for item in nested.iterdir():
                    shutil.move(str(item), destination / item.name)
                nested.rmdir()
        else:
            raise FileNotFoundError(f"Invalid attached source: {source_path}")
    else:
        command = ["git", "clone"]
        if not is_commit:
            command += ["--branch", branch_or_commit, "--depth", "1"]
        command += [git_url, str(destination)]
        subprocess.check_call(command)
    if is_commit and (destination / ".git").is_dir():
        subprocess.check_call(["git", "-C", str(destination), "checkout", branch_or_commit])
    return destination

materialize_source(PROJECT_SOURCE, PROJECT_ROOT, PROJECT_GIT_URL, PROJECT_BRANCH)
materialize_source(ANOMALYCLIP_SOURCE, ANOMALYCLIP_ROOT, ANOMALYCLIP_GIT_URL, ANOMALYCLIP_COMMIT, is_commit=True)
assert (PROJECT_ROOT / "pyproject.toml").is_file(), f"Project checkout is incomplete: {PROJECT_ROOT}"
assert (ANOMALYCLIP_ROOT / "AnomalyCLIP_lib").is_dir(), f"AnomalyCLIP checkout is incomplete: {ANOMALYCLIP_ROOT}"
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", str(PROJECT_ROOT)])
print("Project and dependencies are ready.")


In [ ]:
# Locate attached datasets and optional prior attack_train manifests.
import csv

INPUT_ROOT = Path("/kaggle/input")
selected_datasets = ("mvtec", "visa") if DATASETS == "all" else (DATASETS.lower(),)
if not set(selected_datasets) <= {"mvtec", "visa"}:
    raise ValueError("DATASETS must be all, mvtec, or visa")

def find_mvtec_root():
    matches = sorted({path.parents[2] for path in INPUT_ROOT.rglob("bottle/test/good") if path.is_dir()})
    if len(matches) != 1:
        raise RuntimeError(f"Could not uniquely detect MVTec root; set MVTEC_ROOT. Candidates: {matches}")
    return matches[0]

def find_visa_root():
    matches = sorted({path.parents[1] for path in INPUT_ROOT.rglob("split_csv/1cls.csv") if path.is_file()})
    if len(matches) != 1:
        raise RuntimeError(f"Could not uniquely detect VisA root; set VISA_ROOT. Candidates: {matches}")
    return matches[0]

if "mvtec" in selected_datasets:
    MVTEC_ROOT = Path(MVTEC_ROOT) if MVTEC_ROOT else find_mvtec_root()
if "visa" in selected_datasets:
    VISA_ROOT = Path(VISA_ROOT) if VISA_ROOT else find_visa_root()

def manifest_supports(path, dataset):
    try:
        with Path(path).open("r", newline="", encoding="utf-8-sig") as handle:
            rows = list(csv.DictReader(handle))
        return any(
            str(row.get("dataset", dataset)).lower() == dataset
            and str(row.get("partition", "attack_train")) == "attack_train"
            for row in rows
        )
    except Exception:
        return False

def find_training_manifest(dataset):
    matches = [path for path in INPUT_ROOT.rglob("attack_train_indices.csv") if manifest_supports(path, dataset)]
    if len(matches) == 1:
        return matches[0]
    if len(matches) > 1:
        raise RuntimeError(f"Multiple {dataset} training manifests found; set the corresponding setting explicitly: {matches}")
    return None

if "mvtec" in selected_datasets and not MVTEC_TRAINING_MANIFEST:
    MVTEC_TRAINING_MANIFEST = find_training_manifest("mvtec")
if "visa" in selected_datasets and not VISA_TRAINING_MANIFEST:
    VISA_TRAINING_MANIFEST = find_training_manifest("visa")

print("MVTec root:", MVTEC_ROOT)
print("VisA root:", VISA_ROOT)
print("MVTec training manifest:", MVTEC_TRAINING_MANIFEST or "automatic deterministic split")
print("VisA training manifest:", VISA_TRAINING_MANIFEST or "automatic deterministic split")


In [ ]:
# Build a fully resolved Kaggle configuration.
import yaml

template = PROJECT_ROOT / "configs" / "experiment.example.yaml"
config = yaml.safe_load(template.read_text(encoding="utf-8"))
config["data"]["datasets"] = list(selected_datasets)
config["data"]["mvtec_root"] = str(MVTEC_ROOT) if MVTEC_ROOT else None
config["data"]["visa_root"] = str(VISA_ROOT) if VISA_ROOT else None
config["data"]["mvtec_training_manifest"] = str(MVTEC_TRAINING_MANIFEST) if MVTEC_TRAINING_MANIFEST else None
config["data"]["visa_training_manifest"] = str(VISA_TRAINING_MANIFEST) if VISA_TRAINING_MANIFEST else None
config["model"]["anomalyclip_root"] = str(ANOMALYCLIP_ROOT)
config["model"]["clip_download_root"] = str(WORK_ROOT / "clip_cache")
config["model"]["device"] = "cuda"
if CLIP_WEIGHTS:
    clip_path = Path(CLIP_WEIGHTS)
    if not clip_path.is_file():
        raise FileNotFoundError(f"CLIP_WEIGHTS not found: {clip_path}")
    config["model"]["clip_model_name"] = str(clip_path)
config["artifacts"]["output_root"] = OUTPUT_ROOT
config["artifacts"]["overwrite"] = OVERWRITE
config["training"]["batch_size"] = BATCH_SIZE
config["training"]["show_progress"] = SHOW_PROGRESS

CONFIG_PATH = WORK_ROOT / "kaggle_resolved_input.yaml"
CONFIG_PATH.write_text(yaml.safe_dump(config, sort_keys=False), encoding="utf-8")
print(CONFIG_PATH.read_text(encoding="utf-8"))


In [ ]:
# Mandatory leakage/data preflight. This does not load CLIP.
validation_command = [
    sys.executable, "-m", "object_agnostic_prompt_attack.cli",
    "--config", str(CONFIG_PATH),
    "--datasets", DATASETS,
    "--validate-only",
]
subprocess.check_call(validation_command, cwd=PROJECT_ROOT)


In [ ]:
# Train one independent prompt pair per selected source dataset.
training_command = [
    sys.executable, "-m", "object_agnostic_prompt_attack.cli",
    "--config", str(CONFIG_PATH),
    "--datasets", DATASETS,
]
if OVERWRITE:
    training_command.append("--overwrite")
if RUN_TRAINING:
    subprocess.check_call(training_command, cwd=PROJECT_ROOT)
else:
    print("RUN_TRAINING=False: training skipped after successful validation.")


In [ ]:
# Audit the compact deliverables before download.
import csv, hashlib, json
from object_agnostic_prompt_attack.checkpoint import load_prompt_checkpoint

def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

if RUN_TRAINING:
    for dataset in selected_datasets:
        directory = Path(OUTPUT_ROOT) / dataset
        expected = {"prompts_epoch15.pt", "training_history.csv", "resolved_config.yaml", "manifest.json"}
        actual = {path.name for path in directory.iterdir() if path.is_file()}
        if actual != expected:
            raise RuntimeError(f"Unexpected artifact set for {dataset}: {actual}")
        manifest = json.loads((directory / "manifest.json").read_text(encoding="utf-8"))
        checkpoint = directory / manifest["checkpoint"]["filename"]
        if sha256(checkpoint) != manifest["checkpoint"]["sha256"]:
            raise RuntimeError(f"Checkpoint checksum failed for {dataset}")
        with (directory / "training_history.csv").open(newline="", encoding="utf-8") as handle:
            history = list(csv.DictReader(handle))
        if len(history) != 15:
            raise RuntimeError(f"Expected 15 history rows for {dataset}, got {len(history)}")
        payload = load_prompt_checkpoint(checkpoint)
        if set(payload["prompt_state"]) != {"normal_context", "abnormal_context"}:
            raise RuntimeError(f"Checkpoint is not prompt-only for {dataset}")
        print({
            "dataset": dataset,
            "samples": manifest["sample_count"],
            "epochs": len(history),
            "checkpoint": str(checkpoint),
            "sha256": manifest["checkpoint"]["sha256"],
        })


In [ ]:
# Package all prompt artifacts into one Kaggle-downloadable ZIP.
from IPython.display import FileLink, display

if RUN_TRAINING:
    archive_base = Path("/kaggle/working/object_agnostic_prompt_checkpoints")
    archive_path = Path(shutil.make_archive(str(archive_base), "zip", root_dir=Path(OUTPUT_ROOT).parent, base_dir=Path(OUTPUT_ROOT).name))
    print(f"Created {archive_path} ({archive_path.stat().st_size / 1024**2:.2f} MiB)")
    display(FileLink(str(archive_path)))


## Expected output

The final ZIP contains `prompts/mvtec/` and/or `prompts/visa/`. Each directory has exactly four files: the epoch-15 prompt-only checkpoint, training history, resolved configuration, and reproducibility manifest. Download `object_agnostic_prompt_checkpoints.zip` from the final cell or Kaggle's Output panel.